In [21]:
import os, textwrap
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
load_dotenv(Path("../.env"))


http_proxy = os.getenv("http_proxy")
https_proxy = os.getenv("https_proxy")
HTTP_PROXY = os.getenv("HTTP_PROXY")
HTTPS_PROXY = os.getenv("HTTPS_PROXY")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")

client = Groq(api_key=GROQ_API_KEY)


assert GROQ_API_KEY, "GROQ_API_KEY not found — check that .env exists in the repo root and has a real key."
print("Setup OK")
print(f"  GROQ_MODEL = {GROQ_MODEL}")
print(f"  EMBEDDING_MODEL = {EMBEDDING_MODEL}")

Setup OK
  GROQ_MODEL = openai/gpt-oss-120b
  EMBEDDING_MODEL = sentence-transformers/all-MiniLM-L6-v2


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
DOCUMENT = """
# The Transformer Architecture

## Introduction

The Transformer is a deep learning architecture introduced in the paper "Attention Is All You Need" by Vaswani et al. in 2017. It fundamentally changed the field of natural language processing by replacing recurrent and convolutional layers with a mechanism called self-attention. The model achieved state-of-the-art results on machine translation tasks while being significantly faster to train due to its parallelizable design.

Unlike RNNs, which process tokens sequentially, Transformers process all tokens in a sequence simultaneously. This parallel processing makes them highly efficient on modern hardware accelerators like GPUs and TPUs.

## Self-Attention Mechanism

At the core of every Transformer is the self-attention mechanism. For each token in the input sequence, self-attention computes a weighted sum of all other tokens, where the weights represent how much attention that token should pay to every other token.

The mechanism works by projecting each token into three vectors: Query (Q), Key (K), and Value (V). The attention score between two tokens is computed as the dot product of the query of one token with the key of another, scaled by the square root of the key dimension. These scores are then passed through a softmax to produce attention weights.

Multi-head attention extends this by running several attention operations in parallel, each learning to attend to different aspects of the input — syntactic structure, coreference, semantic roles, and so on.

## Positional Encoding

Because self-attention has no inherent sense of order, Transformers inject positional information into the input embeddings using positional encodings. The original paper used fixed sinusoidal encodings of different frequencies.

Modern variants like BERT and GPT use learned positional embeddings instead, where position representations are parameters trained alongside the rest of the model. More recent models such as RoPE (Rotary Position Embedding) and ALiBi encode position through geometric transformations of the attention scores rather than additive embeddings.

## Feed-Forward Layers

After the attention sub-layer, each Transformer block contains a position-wise feed-forward network (FFN). This consists of two linear transformations with a non-linear activation in between — typically ReLU or GELU.

The FFN is applied independently to each token position, meaning it does not mix information across positions. Research suggests that FFN layers act as key-value memories, storing factual associations learned during training. This is in contrast to attention layers, which primarily perform information routing and aggregation.

## Training and Scale

Transformers are trained using gradient descent with the Adam optimizer, minimizing cross-entropy loss over next-token prediction (in the case of language models) or masked token prediction (in the case of models like BERT).

Scaling has proven to be a dominant lever. Empirical scaling laws (Kaplan et al., 2020) show that model performance improves predictably as a power law with increases in parameters, data, and compute. This insight drove the development of very large models such as GPT-3 (175B parameters), PaLM (540B), and beyond.
"""

In [23]:
def banner(title):
    width = 62
    print(f"\n{'═' * width}")
    print(f"  {title}")
    print(f"{'═' * width}")

def show_chunks(chunks, label="Chunks"):
    print(f"\n{'─' * 62}")
    print(f"  {label}  →  {len(chunks)} chunks")
    print(f"{'─' * 62}")
    for i, chunk in enumerate(chunks, 1):
        text    = chunk.strip()
        preview = text[:110] + ("..." if len(text) > 110 else "")
        print(f"\n  Chunk {i}  ({len(text)} chars)")
        for line in textwrap.wrap(preview, width=58):
            print(f"    {line}")
    avg = sum(len(c) for c in chunks) // len(chunks)
    print(f"\n  Avg chunk size : {avg} chars")
    print(f"{'─' * 62}")

from langchain_huggingface import HuggingFaceEmbeddings

def get_embeddings(texts, model="sentence-transformers/all-MiniLM-L6-v2"):
    """Batch-embed a list of strings. Returns a 2-D numpy array (N × D)."""
    response = client.embeddings.create(input=texts, model=model)
    return np.array([item.embedding for item in response.data])

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

## Fixed-Size Chunking

Fixed-size chunking is the simplest and most deterministic chunking strategy: the document is split every fixed number of characters (400 in this example) with a small overlap of 50 characters to reduce context loss at boundaries. As seen in the output, this approach produces highly uniform chunk sizes and predictable chunk counts, making it useful for quick experiments and token-budget testing. However, because it ignores document structure and linguistic boundaries, several chunks begin or end mid-sentence or even mid-word (for example, “ry (Q), Key (K)...” and “ute. This insight drove...” ). While overlaps partially mitigate abrupt cuts, the resulting chunks often lack semantic coherence compared to more structure-aware approaches.

In [24]:
def fixed_size_chunking(text, chunk_size=400, overlap=50):
    chunks, start = [], 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        # stride = chunk_size - overlap so the next chunk sees
        # the last `overlap` chars of the current one
        start += chunk_size - overlap
    return chunks


banner("Strategy 1 — Fixed-Size Chunking  (chunk=400, overlap=50)")
fixed_chunks = fixed_size_chunking(DOCUMENT, chunk_size=400, overlap=50)
show_chunks(fixed_chunks, "Fixed-Size")


══════════════════════════════════════════════════════════════
  Strategy 1 — Fixed-Size Chunking  (chunk=400, overlap=50)
══════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────
  Fixed-Size  →  10 chunks
──────────────────────────────────────────────────────────────

  Chunk 1  (399 chars)
    # The Transformer Architecture  ## Introduction  The
    Transformer is a deep learning architecture introduced
    in...

  Chunk 2  (400 chars)
    state-of-the-art results on machine translation tasks
    while being significantly faster to train due to its
    par...

  Chunk 3  (400 chars)
    Self-Attention Mechanism  At the core of every Transformer
    is the self-attention mechanism. For each token in ...

  Chunk 4  (400 chars)
    ry (Q), Key (K), and Value (V). The attention score
    between two tokens is computed as the dot product of the
    q...

  Chunk 5  (400 chars)
    ns in parallel, each learning to at

## Recursive Chunking

Recursive chunking improves significantly over fixed-size chunking by attempting to split text at natural boundaries using a hierarchy of separators such as double newlines, single newlines, periods, and finally spaces. Instead of cutting blindly at a character limit, it recursively searches for the best possible split point below the chunk threshold, resulting in chunks that are far more readable and semantically coherent.

In the output, most chunks align naturally with paragraphs or section transitions like “Self-Attention Mechanism” and “Feed-Forward Layers,” preserving the document’s logical flow. However, because the separator hierarchy is manually defined, the strategy still relies on heuristics rather than actual semantic understanding, and chunk sizes can vary considerably — from very small heading-only chunks to larger explanatory sections. Even so, recursive chunking remains one of the most practical and widely used default strategies due to its simplicity, zero API cost, and strong balance between structure preservation and efficiency.

In [25]:
banner("Strategy 2 — Recursive Chunking  (chunk=400, overlap=50)")

splitter = RecursiveCharacterTextSplitter(
    separators   = ["\n\n", "\n", ".", " "],
    chunk_size   = 400,
    chunk_overlap= 50,
    length_function=len,
)
recursive_chunks = splitter.split_text(DOCUMENT)
show_chunks(recursive_chunks, "Recursive")


══════════════════════════════════════════════════════════════
  Strategy 2 — Recursive Chunking  (chunk=400, overlap=50)
══════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────
  Recursive  →  13 chunks
──────────────────────────────────────────────────────────────

  Chunk 1  (47 chars)
    # The Transformer Architecture  ## Introduction

  Chunk 2  (279 chars)
    The Transformer is a deep learning architecture introduced
    in the paper "Attention Is All You Need" by Vaswani...

  Chunk 3  (150 chars)
    . The model achieved state-of-the-art results on machine
    translation tasks while being significantly faster to...

  Chunk 4  (243 chars)
    Unlike RNNs, which process tokens sequentially,
    Transformers process all tokens in a sequence
    simultaneously. ...

  Chunk 5  (283 chars)
    ## Self-Attention Mechanism  At the core of every
    Transformer is the self-attention mechanism. For each
    toke

## Document-Based Chunking

Document-based chunking leverages the document’s inherent structure instead of relying on arbitrary size limits or separator heuristics. In this example, markdown ## headers are used as chunk boundaries, causing each major Transformer topic — such as “Self-Attention Mechanism” or “Training and Scale” — to become its own self-contained chunk. The resulting chunks are highly interpretable and semantically meaningful, closely matching how humans naturally organize and consume technical content. This makes the strategy particularly effective for documentation systems, wikis, research papers, and manuals where headings accurately represent topic boundaries.

In [26]:
import re
def document_based_chunking(text):
    # re.split with a capturing group keeps the delimiter
    parts = re.split(r'(?m)^(##\s+.+)$', text)
    chunks, i = [], 0
    while i < len(parts):
        part = parts[i].strip()
        if part.startswith("##"):
            body = parts[i + 1].strip() if i + 1 < len(parts) else ""
            chunks.append(f"{part}\n\n{body}")
            i += 2
        else:
            if part:
                chunks.append(part)
            i += 1
    return [c for c in chunks if c.strip()]


banner("Strategy 3 — Document-Based Chunking  (split on ## headers)")
doc_chunks = document_based_chunking(DOCUMENT)
show_chunks(doc_chunks, "Document-Based")


══════════════════════════════════════════════════════════════
  Strategy 3 — Document-Based Chunking  (split on ## headers)
══════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────
  Document-Based  →  6 chunks
──────────────────────────────────────────────────────────────

  Chunk 1  (30 chars)
    # The Transformer Architecture

  Chunk 2  (662 chars)
    ## Introduction  The Transformer is a deep learning
    architecture introduced in the paper "Attention Is All
    You...

  Chunk 3  (839 chars)
    ## Self-Attention Mechanism  At the core of every
    Transformer is the self-attention mechanism. For each
    token ...

  Chunk 4  (594 chars)
    ## Positional Encoding  Because self-attention has no
    inherent sense of order, Transformers inject positional
    ...

  Chunk 5  (569 chars)
    ## Feed-Forward Layers  After the attention sub-layer,
    each Transformer block contains a position-wise feed-fo...


## Semantic Chunking

Semantic chunking moves beyond structural heuristics by using embeddings to detect actual shifts in meaning between consecutive sentences. The process first embeds every sentence, computes cosine similarity between neighboring pairs, and then dynamically determines chunk boundaries using a percentile-based threshold rather than a hardcoded similarity value. This is important because embedding models produce different similarity distributions; in this case, text-embedding-3-small generated similarities mostly between 0.2 and 0.6, making a fixed threshold unreliable. By splitting only at the sharpest semantic drops (bottom 25% here), the strategy adapts naturally to the document and embedding model.

In [27]:
def semantic_chunking(text, breakpoint_percentile=25):
    """
    breakpoint_percentile: split at similarity drops in the
    bottom N% of the distribution. Lower = fewer, larger chunks.
    Typical range: 10 (very few splits) to 40 (more splits).
    """
    raw       = re.split(r'(?<=[.?!])\s+', text.strip())
    sentences = [s.strip() for s in raw if len(s.strip()) > 20]

    print(f"\n  Embedding {len(sentences)} sentences...")
    embeddings = get_embeddings(sentences)

    similarities = [
        cosine_similarity(embeddings[i], embeddings[i + 1])
        for i in range(len(embeddings) - 1)
    ]

    # Dynamic threshold: bottom Nth percentile of similarities
    threshold = np.percentile(similarities, breakpoint_percentile)

    print(f"\n  Similarity distribution:")
    print(f"    Min   : {min(similarities):.3f}")
    print(f"    Max   : {max(similarities):.3f}")
    print(f"    Mean  : {np.mean(similarities):.3f}")
    print(f"    Threshold (p{breakpoint_percentile}): {threshold:.3f}  ← split below this")

    print(f"\n  Cosine similarity between consecutive sentences:")
    print(f"  (dynamic threshold = {threshold:.3f}  |  below → chunk boundary)")
    for i, sim in enumerate(similarities):
        marker = "  ← SPLIT" if sim < threshold else ""
        print(f"    [{i:02d}→{i+1:02d}]  {sim:.3f}{marker}")

    chunks, current = [], [sentences[0]]
    for i, sim in enumerate(similarities):
        if sim < threshold:
            chunks.append(" ".join(current))
            current = []
        current.append(sentences[i + 1])
    if current:
        chunks.append(" ".join(current))

    return chunks


banner("Strategy 4 — Semantic Chunking  (dynamic threshold, p25)")
semantic_chunks = semantic_chunking(DOCUMENT, breakpoint_percentile=25)
show_chunks(semantic_chunks, "Semantic")


══════════════════════════════════════════════════════════════
  Strategy 4 — Semantic Chunking  (dynamic threshold, p25)
══════════════════════════════════════════════════════════════

  Embedding 24 sentences...


NotFoundError: Error code: 404 - {'error': {'message': 'The model `sentence-transformers/all-minilm-l6-v2` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}